# 03 - Synthetic evidence documents (SPEC sections 8.4 and 9.1)
Creates `num_evidence_docs` PDF evidence items for the open cases (emails, bank statements, board minutes, transcripts, contracts, witness statements). Each document names the real respondent company, its synthetic directors, amounts and dates. Background passages are adapted from real *ASIC v ...* judgments (P10) so the text reads like real evidence.

Output:
* PDFs: `/Volumes/<catalog>/<schema>/<evidence_volume>/<case_id>/<evidence_id>.pdf` (originals are never modified)
* Manifest: `/Volumes/<catalog>/<schema>/<raw_volume>/synthetic/evidence_manifest/` (custodian, source system, document type, SHA-256 at creation, source citation)

The pipeline parses these files with `ai_parse_document`. Every document is stamped SYNTHETIC.

In [0]:
%pip install -q fpdf2
dbutils.library.restartPython()

In [0]:
dbutils.widgets.text("catalog", "")
dbutils.widgets.text("schema", "")
dbutils.widgets.text("raw_volume", "raw")
dbutils.widgets.text("evidence_volume", "evidence")
dbutils.widgets.text("num_evidence_docs", "500")
dbutils.widgets.text("random_seed", "42")

CATALOG = dbutils.widgets.get("catalog")
SCHEMA = dbutils.widgets.get("schema")
RAW_ROOT = f"/Volumes/{CATALOG}/{SCHEMA}/{dbutils.widgets.get('raw_volume')}"
EVID_ROOT = f"/Volumes/{CATALOG}/{SCHEMA}/{dbutils.widgets.get('evidence_volume')}"
NUM_DOCS = int(dbutils.widgets.get("num_evidence_docs"))
SEED = int(dbutils.widgets.get("random_seed"))
assert CATALOG and SCHEMA

In [0]:
import os, re, json, random, hashlib, datetime as dt
from pyspark.sql import functions as F

rng = random.Random(SEED)
syn = lambda d: spark.read.parquet(f"{RAW_ROOT}/synthetic/{d}")
cases = syn("cases").join(syn("curated_slice").select(F.col("acn").alias("respondent_acn"), "company_name"), "respondent_acn").toPandas()
parties = (syn("case_parties").filter("party_kind = 'person'")
           .join(syn("persons").select(F.col("person_id").alias("party_ref"), "given_name", "family_name"), "party_ref")
           .toPandas())
users = syn("users").filter("persona = 'Investigator'").toPandas()

# Background excerpts from real ASIC v ... judgments (P10); fall back to neutral text if P10 is unavailable
excerpts = []
p10 = f"{RAW_ROOT}/public/p10_open_au_legal_corpus"
if os.path.exists(p10):
    snap = sorted(d.name for d in os.scandir(p10) if d.name.startswith("snapshot="))[-1]
    with open(f"{p10}/{snap}/asic_judgments.jsonl") as f:
        for line in f:
            doc = json.loads(line)
            paras = [p.strip() for p in re.split(r"\n\s*\n", doc.get("text") or "") if 400 < len(p.strip()) < 1500]
            if paras:
                excerpts.append((doc.get("citation"), rng.choice(paras)))
if not excerpts:
    excerpts = [(None, "The respondent is alleged to have engaged in conduct that was misleading or deceptive, and its officers are alleged to have failed to exercise their powers with the degree of care and diligence required.")]
print(len(cases), "cases,", len(parties), "person parties,", len(excerpts), "judgment excerpts")

In [0]:
DOC_TYPES = [("Email", 30), ("Bank statement", 15), ("Board minutes", 12), ("Interview transcript", 12),
             ("Contract", 10), ("Witness statement", 11), ("Spreadsheet export", 10)]
SOURCES = [("Nuix export", 70), ("Direct upload", 20), ("s33 notice production", 10)]

def wchoice(pairs):
    return rng.choices([p[0] for p in pairs], weights=[p[1] for p in pairs])[0]

def money():
    return round(rng.lognormvariate(11, 1.4), -2)

def body(doc_type, case, people, when):
    co, acn = case["company_name"], case["respondent_acn"]
    p1 = people[0] if people else "the director"
    p2 = people[1] if len(people) > 1 else "the finance manager"
    amt, amt2 = money(), money()
    acct = f"{rng.randint(100, 999)}-{rng.randint(100, 999)} {rng.randint(10000000, 99999999)}"
    if doc_type == "Email":
        return (f"From: {p1}\nTo: {p2}\nDate: {when:%d %B %Y %H:%M}\nSubject: Transfer before the administrators arrive\n\n"
                f"{p2.split()[0]}, please move ${amt:,.0f} from the {co} operating account ({acct}) to the new entity today. "
                f"Do not record it as a loan. We will re-invoice the customers from the new company next week.")
    if doc_type == "Bank statement":
        rows = "\n".join(f"{(when + dt.timedelta(days=i)):%d/%m/%Y}  TRANSFER OUT  ${money():,.2f}" for i in range(6))
        return f"Account holder: {co} (ACN {acn})\nAccount: {acct}\nStatement period ending {when:%d %B %Y}\n\n{rows}"
    if doc_type == "Board minutes":
        return (f"Minutes of a meeting of directors of {co} (ACN {acn}) held on {when:%d %B %Y}.\nPresent: {', '.join(people[:3]) or p1}.\n"
                f"1. The board noted ATO debts of approximately ${amt:,.0f}.\n2. Resolved to transfer customer contracts and plant valued at ${amt2:,.0f} for nominal consideration.\n"
                f"3. Noted that solvency had not been assessed.")
    if doc_type == "Interview transcript":
        return (f"Record of examination under s19 ASIC Act. Examinee: {p1}. Date: {when:%d %B %Y}.\n\n"
                f"Q: Were you aware the company could not pay its debts?\nA: I relied on the accountant.\n"
                f"Q: Who authorised the payment of ${amt:,.0f} on that date?\nA: I signed it, but {p2} prepared it.")
    if doc_type == "Contract":
        return (f"Asset Sale Agreement dated {when:%d %B %Y} between {co} (ACN {acn}) (Vendor) and a related purchaser.\n"
                f"Purchase price: ${amt2 / 50:,.0f}. Assets: customer list, equipment, goodwill. Signed by {p1}.")
    if doc_type == "Witness statement":
        return (f"Statement of {p2}, dated {when:%d %B %Y}.\n1. I was employed by {co}.\n"
                f"2. I was told by {p1} that customers would be moved to a new company.\n3. I estimate customers lost ${amt:,.0f} in prepaid services.")
    return (f"Export of ledger for {co} (ACN {acn}) as at {when:%d %B %Y}.\nCreditors: ATO ${amt:,.0f}; trade creditors ${amt2:,.0f}.\n"
            f"Related party loan account ({p1}): ${money():,.0f}.")

In [0]:
from fpdf import FPDF

def latin(s):
    return (s or "").replace("\u2019", "'").replace("\u2013", "-").replace("\u2014", "-").encode("latin-1", "replace").decode("latin-1")

def render(title, text, excerpt, citation):
    pdf = FPDF(); pdf.set_creation_date(FIXED_CREATION)  # deterministic bytes -> stable SHA-256 across reruns
    pdf.add_page(); pdf.set_auto_page_break(True, 15)
    pdf.set_font("Helvetica", "B", 9); pdf.cell(0, 5, "SYNTHETIC DEMONSTRATION DOCUMENT - NOT REAL EVIDENCE", new_x="LMARGIN", new_y="NEXT")
    pdf.set_font("Helvetica", "B", 13); pdf.multi_cell(0, 7, latin(title))
    pdf.set_font("Helvetica", size=10); pdf.ln(2); pdf.multi_cell(0, 5, latin(text))
    if excerpt:
        pdf.ln(4); pdf.set_font("Helvetica", "I", 9)
        pdf.multi_cell(0, 4.5, latin(f"Background (adapted from public judgment {citation or 'n/a'}): {excerpt}"))
    return bytes(pdf.output())

FIXED_CREATION = dt.datetime(2026, 1, 1, tzinfo=dt.timezone.utc)
manifest, kept_existing = [], 0
for i in range(NUM_DOCS):
    case = cases.iloc[i % len(cases)]
    people = [f"{r.given_name} {r.family_name}" for r in parties[parties.case_id == case.case_id].itertuples()]
    rng.shuffle(people)
    doc_type = wchoice(DOC_TYPES)
    when = case.opened_at.to_pydatetime() - dt.timedelta(days=rng.randint(30, 900), minutes=rng.randint(0, 1440))
    citation, excerpt = rng.choice(excerpts) if rng.random() < 0.6 else (None, None)
    evidence_id = f"E{i:06d}"
    filename = f"{evidence_id}_{doc_type.lower().replace(' ', '_')}.pdf"
    data = render(f"{doc_type} - {case.company_name}", body(doc_type, case, people, when), excerpt, citation)
    path = f"{EVID_ROOT}/{case.case_id}/{filename}"
    os.makedirs(os.path.dirname(path), exist_ok=True)
    if os.path.exists(path):
        # Chain of custody: originals are never modified. Hash the file that is already in the volume.
        with open(path, "rb") as f:
            data = f.read()
        kept_existing += 1
    else:
        with open(path, "wb") as f:
            f.write(data)
    manifest.append((evidence_id, case.case_id, filename, path, doc_type, wchoice(SOURCES),
                     rng.choice(users.user_id.tolist()), when, hashlib.sha256(data).hexdigest(), citation))

m = spark.createDataFrame(manifest, "evidence_id string, case_id string, filename string, path string, doc_type string, source_system string, "
                                    "custodian string, document_date timestamp, sha256_at_creation string, source_citation string")
(m.withColumn("provenance", F.lit("synthetic")).withColumn("generated_at", F.current_timestamp())
  .write.mode("overwrite").parquet(f"{RAW_ROOT}/synthetic/evidence_manifest"))
assert spark.read.parquet(f"{RAW_ROOT}/synthetic/evidence_manifest").count() == NUM_DOCS
print(f"{NUM_DOCS - kept_existing} new documents written, {kept_existing} existing originals left untouched")
display(m.groupBy("doc_type").count())